# 🔍 Advanced RAG — Hybrid Search
### Keyword + Vector Search + Ensemble Retrieval
**by Mayank Chugh | IT AI Enthusiast | @itaienthusiast**

---
### 📋 Notebook Structure

| Section | Topic |
|---------|-------|
| 1 | What is Hybrid Search? |
| 2 | Why Naive Retrieval Isn't Enough |
| 3 | Hybrid RAG Architecture |
| 4 | Keyword Search: TF-IDF from Scratch |
| 5 | Keyword Search: Ranking |
| 6 | Vector Search: Dense Embeddings |
| 7 | The Librarian Analogy |
| 8 | Pipeline Setup: PDF → Chunks → ChromaDB |
| 9 | BM25 Retriever + Ensemble Retriever |
| 10 | Alpha Weighting Explained |
| 11 | Load Quantized LLM (Zephyr 7B, 4-bit) |
| 12 | Naive vs Hybrid: Results Comparison |
| 13 | Key Takeaways |

---
**Runtime:** GPU required (Colab → Runtime → Change runtime type → T4 GPU)  
**Key Libraries:** `langchain`, `chromadb`, `rank_bm25`, `sentence-transformers`, `bitsandbytes`


## ⚙️ Install Dependencies


In [1]:
# Install all required packages
!pip install -q pypdf langchain langchain_community langchain_text_splitters chromadb rank_bm25 bitsandbytes accelerate sentence-transformers langchain_classic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.

In [2]:
# Upgrade LangChain packages to ensure the latest structure and fixes
!pip install -q --upgrade langchain langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.4/245.4 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.5/160.5 kB 7.5 MB/s eta 0:00:00


---
# 🧠 — `What is Hybrid Search?`
---


Hybrid Search combines two complementary retrieval strategies:

| Approach | Algorithm | Vector Type | Strength |
|----------|-----------|-------------|----------|
| **Keyword Search** | TF-IDF / BM25 | Sparse (mostly zeros) | Exact term matching |
| **Vector Search** | Transformer Embeddings | Dense (all values filled) | Semantic / conceptual similarity |
| **Hybrid (Ensemble)** | Weighted fusion | Both | Best of both worlds |

**Core formula:**
```
Hybrid Score = α × Dense Score  +  (1 - α) × Sparse Score
```
> When `α = 0.3`: 30% vector + 70% keyword (technical doc mode)  
> When `α = 0.5`: Equal blend  
> When `α = 1.0`: Pure vector search only


---
# ⚠️ — `Why Naive Retrieval Isn't Enough`
---


Most RAG tutorials only show **dense vector similarity** — but this has a blind spot:

- If your query uses exact technical terms, semantic search **dilutes** them with related-but-imprecise results
- If the document uses the exact word you need, keyword search **finds it directly**
- Neither alone is sufficient for production RAG

**RAG Retrieval Techniques roadmap** (this series):

1. ✅ Naive Retrieval — baseline (covered in previous episodes)
2. ▶️ **Hybrid Search — THIS NOTEBOOK**
3. ⬜ Sentence Window Retrieval
4. ⬜ Self-Query Retrieval
5. ⬜ Parent Document Retrieval
6. ⬜ HyDE (Hypothetical Document Embeddings)


---
# 🏗️ — `Hybrid RAG Architecture
---


```
┌─────────────────────────────────────────────────────────────────────┐
│                    HYBRID RAG PIPELINE                              │
├─────────────────┬───────────────────────────┬───────────────────────┤
│   Phase 1       │        Phase 2            │      Phase 3          │
│   INDEX         │        RETRIEVE           │      GENERATE         │
│                 │                           │                       │
│  PDF/Document   │  User Query               │  Top-K Context Docs   │
│       │         │       ├─────────────────┐ │         │             │
│  Text Splitter  │  Dense Retriever  BM25  │ │  Quantized LLM        │
│  chunk=200      │  (vector sim.)    Kw.   │ │  Zephyr 7B (4-bit)    │
│  overlap=30     │       └──────┬────┘     │ │         │             │
│       │         │        Ensemble         │ │  Hybrid RAG Answer    │
│  BGE Embedding  │   (α=0.3 vec + 0.7 kw)  │ │                       │
│       │         │              │          │ │                       │
│  ChromaDB       │      Ranked Top-K Docs  │ │                       │
│  Vector Store   │                         │ │                       │
└─────────────────┴───────────────────────────┴───────────────────────┘
```

**Key difference from naive RAG:** Phase 2 runs **two parallel retrieval paths** whose scores are blended before ranking.


---
# 🔑 — `Keyword Search: TF-IDF from Scratch`
---


Before using LangChain's BM25Retriever, let's **build keyword search from scratch** so the mechanics are clear.

### What is TF-IDF?
```
TF  = (# times term appears in document) / (total # terms in document)
IDF = log( total documents / # documents containing the term )
TF-IDF Score = TF × IDF
```
- Common words like `the`, `is` → **low IDF** (appear everywhere) → low score  
- Rare domain words like `keyword`, `sparse` → **high IDF** → high score  
- Result: a **sparse vector** — most values are zero, only vocabulary words present in the doc score non-zero


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import re


#### Sample Documents — Our Mini Corpus


In [4]:
# Four sample documents — notice 'keyword' appears multiple times across them
documents = [
    "This is a list that includes a few sample documents for demonstration purposes.",
    "Keywords are essential because they power keyword-based search and retrieval.",
    "Document analysis often focuses on extracting meaningful keywords and phrases.",
    "Keyword-based search typically relies on sparse embeddings to match relevant terms.",
    "Using clear keywords can improve search accuracy and help find the most relevant documents."
]

query = "keyword-based search"
print(f"Query: '{query}'")
print(f"\nDocuments ({len(documents)} total):")
for i, doc in enumerate(documents):
    print(f"  Doc {i}: {doc}")


Query: 'keyword-based search'

Documents (5 total):
  Doc 0: This is a list that includes a few sample documents for demonstration purposes.
  Doc 1: Keywords are essential because they power keyword-based search and retrieval.
  Doc 2: Document analysis often focuses on extracting meaningful keywords and phrases.
  Doc 3: Keyword-based search typically relies on sparse embeddings to match relevant terms.
  Doc 4: Using clear keywords can improve search accuracy and help find the most relevant documents.


#### Step 1 — Pre-process: Lowercase + Remove Punctuation
> Pre-processing prevents case-sensitivity issues: `Keyword` ≠ `keyword` without this step.


In [5]:
def preprocess_text(text):
    """Lowercase and strip punctuation so 'Keyword' == 'keyword'."""
    text = text.lower()
    text = re.sub(r'[^ɑ\w\s]', '', text)  # remove punctuation
    return text

preprocessed_docs = [preprocess_text(doc) for doc in documents]
preprocessed_query = preprocess_text(query)

print("Preprocessed Documents:")
for i, doc in enumerate(preprocessed_docs):
    print(f"  [{i}] {doc}")
print(f"\nPreprocessed Query: '{preprocessed_query}'")

Preprocessed Documents:
  [0] this is a list that includes a few sample documents for demonstration purposes
  [1] keywords are essential because they power keywordbased search and retrieval
  [2] document analysis often focuses on extracting meaningful keywords and phrases
  [3] keywordbased search typically relies on sparse embeddings to match relevant terms
  [4] using clear keywords can improve search accuracy and help find the most relevant documents

Preprocessed Query: 'keywordbased search'


#### Step 2 — Build TF-IDF Sparse Vectors
> `fit_transform` builds the vocabulary from the corpus, then converts each document into a TF-IDF vector.


In [6]:
import pandas as pd

# Build TF-IDF vectors for all documents
vectorizer = TfidfVectorizer()
doc_vectors = vectorizer.fit_transform(preprocessed_docs)  # sparse matrix: docs × vocab

# Show vocabulary
vocab = vectorizer.get_feature_names_out()
print(f"Vocabulary ({len(vocab)} unique words): {list(vocab)}")

# Show the sparse matrix as a dense array
print(f"\nTF-IDF Matrix shape: {doc_vectors.shape}  (docs × vocab_size)")
print("\nTF-IDF Matrix (rows=documents, cols=vocabulary words):")
df_tfidf = pd.DataFrame(doc_vectors.toarray(),
                         columns=vocab,
                         index=[f"Doc {i}" for i in range(len(documents))])
print(df_tfidf.to_string())
print("\n💡 Most values are 0 — this is what 'sparse vector' means.")

Vocabulary (46 unique words): ['accuracy', 'analysis', 'and', 'are', 'because', 'can', 'clear', 'demonstration', 'document', 'documents', 'embeddings', 'essential', 'extracting', 'few', 'find', 'focuses', 'for', 'help', 'improve', 'includes', 'is', 'keywordbased', 'keywords', 'list', 'match', 'meaningful', 'most', 'often', 'on', 'phrases', 'power', 'purposes', 'relevant', 'relies', 'retrieval', 'sample', 'search', 'sparse', 'terms', 'that', 'the', 'they', 'this', 'to', 'typically', 'using']

TF-IDF Matrix shape: (5, 46)  (docs × vocab_size)

TF-IDF Matrix (rows=documents, cols=vocabulary words):
       accuracy  analysis       and       are   because       can     clear  demonstration  document  documents  embeddings  essential  extracting       few      find   focuses       for      help   improve  includes        is  keywordbased  keywords      list     match  meaningful      most     often        on   phrases     power  purposes  relevant    relies  retrieval    sample    search    

#### Step 3 — Encode the Query & Compute Cosine Similarity


In [7]:
# Encode the query using the SAME vocabulary (transform only, NOT fit_transform)
query_vector = vectorizer.transform([preprocessed_query])

print("Query vector (sparse):")
df_query = pd.DataFrame(query_vector.toarray(), columns=vocab, index=["Query"])
# Show only non-zero columns
non_zero_cols = df_query.columns[(df_query != 0).any()]
print(df_query[non_zero_cols].to_string())

# Cosine similarity: angle between document vector and query vector
similarities = cosine_similarity(doc_vectors, query_vector)  # shape: (4, 1)
print("\nCosine Similarities (query vs each document):")
for i, score in enumerate(similarities.flatten()):
    print(f"  Doc {i} ({documents[i][:40]}...): {score:.4f}")

Query vector (sparse):
       keywordbased    search
Query      0.769447  0.638711

Cosine Similarities (query vs each document):
  Doc 0 (This is a list that includes a few sampl...): 0.0000
  Doc 1 (Keywords are essential because they powe...): 0.3708
  Doc 2 (Document analysis often focuses on extra...): 0.0000
  Doc 3 (Keyword-based search typically relies on...): 0.3420
  Doc 4 (Using clear keywords can improve search ...): 0.1253


#### Step 4 — Rank Documents by Similarity Score


In [8]:
# Sort indices by similarity score — DESCENDING (highest first)
ranked_indices = np.argsort(similarities, axis=0)[::-1].flatten()

print(f"Ranked document indices (best → worst match): {ranked_indices}")
print(f"\nQuery: '{query}'")
print("\nKeyword Search Results — Ranked by TF-IDF Cosine Similarity:")
print("-" * 60)
for rank, idx in enumerate(ranked_indices):
    score = similarities.flatten()[idx]
    print(f"Rank {rank+1} (score={score:.4f}): {documents[idx]}")

print("\n💡 Notice: Doc 1 and Doc 3 rank highest because 'keyword' appears")
print("   multiple times — higher TF × non-zero IDF = higher TF-IDF score.")

Ranked document indices (best → worst match): [1 3 4 2 0]

Query: 'keyword-based search'

Keyword Search Results — Ranked by TF-IDF Cosine Similarity:
------------------------------------------------------------
Rank 1 (score=0.3708): Keywords are essential because they power keyword-based search and retrieval.
Rank 2 (score=0.3420): Keyword-based search typically relies on sparse embeddings to match relevant terms.
Rank 3 (score=0.1253): Using clear keywords can improve search accuracy and help find the most relevant documents.
Rank 4 (score=0.0000): Document analysis often focuses on extracting meaningful keywords and phrases.
Rank 5 (score=0.0000): This is a list that includes a few sample documents for demonstration purposes.

💡 Notice: Doc 1 and Doc 3 rank highest because 'keyword' appears
   multiple times — higher TF × non-zero IDF = higher TF-IDF score.


---
# 🧬 — Vector Search: Dense Embeddings (Conceptual Demo)`
---


Dense embeddings encode the **full semantic meaning** of a sentence into a fixed-size vector where **every dimension has a value**.

In practice these come from models like `BAAI/bge-base-en-v1.5` (768 dims). Below we use toy 5-dim vectors to demonstrate the mechanics.

> **Sentence Transformer models:** https://huggingface.co/sentence-transformers  
> Used in the full pipeline: `BAAI/bge-base-en-v1.5`


In [9]:
# Toy 5-dimensional dense vectors (simulating transformer embeddings)
# In production: shape would be (n_docs, 768) from BGE-base
doc_embeddings = np.array([
    [0.634, 0.234, 0.867, 0.042, 0.249],  # Doc 0
    [0.123, 0.456, 0.789, 0.321, 0.654],  # Doc 1
    [0.987, 0.654, 0.321, 0.123, 0.456],  # Doc 2
])

query_dense = np.array([[0.789, 0.321, 0.654, 0.987, 0.123]])

print("Dense Document Embeddings (shape:", doc_embeddings.shape, "):")
print("  Note: ALL values are non-zero — this is what 'dense vector' means")
for i, emb in enumerate(doc_embeddings):
    print(f"  Doc {i}: {emb}")

# Cosine similarity between dense query and dense docs
dense_similarities = cosine_similarity(doc_embeddings, query_dense)
ranked_dense = np.argsort(dense_similarities, axis=0)[::-1].flatten()

print("\nVector Search Results — Ranked by Dense Cosine Similarity:")
print("-" * 50)
for rank, idx in enumerate(ranked_dense):
    score = dense_similarities.flatten()[idx]
    print(f"Rank {rank+1} (score={score:.4f}): Document {idx+1}")

print("\n💡 KEY DIFFERENCE: Sparse (keyword) uses exact word matching.")
print("   Dense uses learned semantic features — similar meaning, not same words.")


Dense Document Embeddings (shape: (3, 5) ):
  Note: ALL values are non-zero — this is what 'dense vector' means
  Doc 0: [0.634 0.234 0.867 0.042 0.249]
  Doc 1: [0.123 0.456 0.789 0.321 0.654]
  Doc 2: [0.987 0.654 0.321 0.123 0.456]

Vector Search Results — Ranked by Dense Cosine Similarity:
--------------------------------------------------
Rank 1 (score=0.7356): Document 1
Rank 2 (score=0.7152): Document 3
Rank 3 (score=0.6736): Document 2

💡 KEY DIFFERENCE: Sparse (keyword) uses exact word matching.
   Dense uses learned semantic features — similar meaning, not same words.


---
# 📚 — `The Two-Librarian Analogy`
---


| | Librarian A (Keyword) | Librarian B (Vector) | Hybrid |
|--|--|--|--|
| **Method** | Card index — exact words | Read every book — understands concepts | Both work simultaneously |
| **Finds 'Sunny'?** | ✅ Yes | ✅ Yes | ✅ Yes |
| **Finds 'Bunny' when searching 'Sunny'?** | ❌ Not in index | ✅ Phonetically/conceptually close | ✅ Yes |
| **Finds exact technical term?** | ✅ Perfect | ⚠️ May dilute | ✅ Yes |
| **Handles synonyms/paraphrases?** | ❌ No | ✅ Yes | ✅ Yes |

**The Ensemble Retriever is the manager who takes both librarians' ranked lists and produces a blended final ranking using alpha weights.**


---
# 📄 — `Full Pipeline: PDF → Chunks → Embeddings → ChromaDB`
---


Now we move from the conceptual demo to the **real pipeline** using the RAG research paper.

**Document:** `Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks`  
**Upload this PDF to Colab** before running the cells below.


In [10]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Load PDF ──────────────────────────────────────────────────────────────
doc_path = "/content/NeurIPS-2020-retrieval-augmented-generation-for-knowledge-intensive-nlp-tasks-Paper.pdf"  # adjust if needed
loader = PyPDFLoader(doc_path)
docs = loader.load()

/tmp/ipykernel_3213/2716970411.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [11]:
# ── Chunk: 200 tokens, 30 token overlap ──────────────────────────────────
# chunk_size=200: each chunk is ~200 characters
# chunk_overlap=30: last 30 chars of chunk N become first 30 of chunk N+1
#   → prevents context from being cut off at chunk boundaries
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = splitter.split_documents(docs)
print(f"Created {len(chunks)} chunks")
print(f"\nSample chunk (first):")
print(chunks[0].page_content)


Created 368 chunks

Sample chunk (first):
Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡ , Ethan Perez?,
Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,


#### Embedding Model: BAAI/bge-base-en-v1.5
> Open-source, 768-dimensional dense embeddings. Available via HuggingFace Inference API.  
> Get your free token at: https://huggingface.co/settings/tokens


In [12]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Use local HuggingFaceEmbeddings (sentence-transformers) instead of Inference API
# This runs locally — no API token or internet call required
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)
print("Embedding model loaded: BAAI/bge-base-en-v1.5 (768-dim dense vectors)")

/tmp/ipykernel_3213/1540737347.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded: BAAI/bge-base-en-v1.5 (768-dim dense vectors)


In [13]:
# ── Store in ChromaDB (in-memory) ────────────────────────────────────────
# ChromaDB is used here in in-memory mode — data is lost when runtime resets.
# For persistent storage: Chroma(persist_directory='./chroma_db', ...)
vectorstore = Chroma.from_documents(chunks, embeddings)

# Dense retriever: returns top 3 results by cosine similarity
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("ChromaDB vectorstore created.")
print(f"Dense retriever ready — returns top-3 chunks by vector similarity.")


ChromaDB vectorstore created.
Dense retriever ready — returns top-3 chunks by vector similarity.


---
# ⚖️ — `BM25 Retriever + Ensemble Retriever`
---


**BM25** (Best Match 25) is the production-grade evolution of TF-IDF:
- Adds **document length normalisation** — longer docs don't unfairly score higher
- Adds **term frequency saturation** — word appearing 50× doesn't score 50× higher than 2×
- Default algorithm in Elasticsearch


In [14]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# ── BM25 Keyword Retriever ────────────────────────────────────────────────
# Built directly from the same chunks — no embedding model needed
# BM25 builds a sparse keyword index from the document text
keyword_retriever = BM25Retriever.from_documents(chunks)
keyword_retriever.k = 3  # return top 3 by keyword match
print("BM25 keyword retriever ready — returns top-3 chunks by keyword match.")

BM25 keyword retriever ready — returns top-3 chunks by keyword match.


---
# 🎛️ — `Alpha Weighting — Tuning the Ensemble`
---


```
Hybrid Score = α × Dense Score  +  (1 - α) × Sparse Score
```

| weights=[dense, keyword] | Dense % | Keyword % | Best for |
|--------------------------|---------|-----------|----------|
| `[1.0, 0.0]` | 100% | 0% | Pure semantic / conversational |
| `[0.5, 0.5]` | 50% | 50% | General-purpose RAG |
| `[0.3, 0.7]` | 30% | 70% | **Technical docs (this demo)** |
| `[0.0, 1.0]` | 0% | 100% | Pure keyword / exact-match |

> We use `[0.3, 0.7]` here — the research paper contains precise technical terms that BM25 handles better.


In [15]:
# ── Ensemble Retriever: blend dense + keyword with alpha weights ──────────
# weights=[0.3, 0.7] means:
#   30% weight to dense_retriever (vector similarity)
#   70% weight to keyword_retriever (BM25 sparse)
#
# Internally uses Reciprocal Rank Fusion (RRF):
#   RRF_score = sum(weight / (rank + 60)) for each retriever

ensemble_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, keyword_retriever],
    weights=[0.3, 0.7]
)
print("Ensemble retriever ready.")
print("  Retriever 1 (Dense)   → 30% weight → BGE embeddings + ChromaDB")
print("  Retriever 2 (Keyword) → 70% weight → BM25 sparse index")
print("  Fusion: Reciprocal Rank Fusion (RRF)")


Ensemble retriever ready.
  Retriever 1 (Dense)   → 30% weight → BGE embeddings + ChromaDB
  Retriever 2 (Keyword) → 70% weight → BM25 sparse index
  Fusion: Reciprocal Rank Fusion (RRF)


---
# 🤖 — `Load Quantized LLM: Zephyr 7B (4-bit)`
---


We use **Zephyr 7B** loaded at **4-bit precision** (NF4 quantization via BitsAndBytes):
- Reduces memory from ~14GB (FP16) to ~4GB → fits on free Colab T4 GPU
- NF4 (NormalFloat4) preserves the normal distribution of weights → better quality than INT4
- Less than ~2% quality degradation at 75% memory saving

> ⚠️ **Requires GPU runtime.** Go to Runtime → Change runtime type → T4 GPU


In [16]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from langchain_community.llms import HuggingFacePipeline

model_name = "HuggingFaceH4/zephyr-7b-beta"


In [17]:
def load_quantized_model(model_name: str):
    """Load a causal LLM in 4-bit NF4 quantization to fit in ~4GB GPU memory."""
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,   # nested quantization for extra compression
        bnb_4bit_quant_type="nf4",         # NormalFloat4 — best quality at 4-bit
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        quantization_config=bnb_config,
    )
    return model

def initialize_tokenizer(model_name: str):
    """Load model-specific tokenizer. Each model has its own vocabulary."""
    tokenizer = AutoTokenizer.from_pretrained(model_name, return_token_type_ids=False)
    tokenizer.bos_token_id = 1  # beginning-of-sentence token id
    return tokenizer


In [18]:
# ── Load tokenizer and model (takes 3-5 minutes on Colab T4) ─────────────
tokenizer = initialize_tokenizer(model_name)
model = load_quantized_model(model_name)
print(f"Model loaded: {model_name}")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")


config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

...

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded: HuggingFaceH4/zephyr-7b-beta
GPU memory used: 4.57 GB


In [19]:
# ── Build text-generation pipeline ────────────────────────────────────────
hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    use_cache=True,
    device_map="auto",
    max_length=2048,
    do_sample=True,
    top_k=5,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)
print("LLM pipeline ready.")


Passing `generation_config` together with generation-related arguments=({'use_cache', 'do_sample', 'num_return_sequences', 'eos_token_id', 'max_length', 'pad_token_id', 'top_k'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM pipeline ready.


/tmp/ipykernel_3213/471252132.py:16: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=hf_pipeline)


---
# 🔬 — `Naive vs Hybrid: Results Comparison`
---


We build **two chains** using the same LLM but different retrievers:

| Chain | Retriever | Docs Retrieved | Type |
|-------|-----------|----------------|------|
| `normal_chain` | ChromaDB (dense only) | 3 | Dense vector similarity |
| `hybrid_chain` | EnsembleRetriever | 6 (3+3) | Dense + BM25 keyword |


In [20]:
from langchain_classic.chains import RetrievalQA

# ── Chain 1: Naive RAG — dense retriever only ─────────────────────────────
normal_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",          # 'stuff' = pack all retrieved docs into the prompt
    retriever=dense_retriever
)

# ── Chain 2: Hybrid RAG — ensemble retriever ──────────────────────────────
hybrid_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=ensemble_retriever
)

print("Both chains ready.")
print("  normal_chain → dense_retriever (3 docs, vector only)")
print("  hybrid_chain → ensemble_retriever (up to 6 docs, dense + BM25)")


Both chains ready.
  normal_chain → dense_retriever (3 docs, vector only)
  hybrid_chain → ensemble_retriever (up to 6 docs, dense + BM25)


#### Test Query 1: RAG Token Model


In [21]:
query = "What is the RAG token model?"
print(f"Query: '{query}'\n")
print("=" * 60)
print("NAIVE RAG (dense only):")
print("=" * 60)
response1 = normal_chain.invoke(query)
print(response1.get("result"))


Query: 'What is the RAG token model?'

NAIVE RAG (dense only):


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

RAG-Token The RAG-Token model can be seen as a standard, autoregressive seq2seq genera-
tor with transition probability: p0
✓(yi|x, y1:i  1)= P
z2top-k(p(·|x)) p⌘ (zi|x)p✓(yi|x, zi,y 1:i  1) To

distribution over generated text. In one approach, RAG-Sequence, the model uses the same document
to predict each target token. The second approach, RAG-Token, can predict each target token based

NY
i
p✓(yi|x, z, y1:i  1)
RAG-Token Model In the RAG-Token model we can draw a different latent document for each

Question: What is the RAG token model?
Helpful Answer: The RAG token model is an autoregressive seq2seq generator that predicts each target token based on the input document and the previous generated tokens. It differs from the RAG sequence model, which uses the same document to predict each target token. In the RAG token mode

In [22]:
print("=" * 60)
print("HYBRID RAG (dense + BM25, weights=[0.3, 0.7]):")
print("=" * 60)
response2 = hybrid_chain.invoke(query)
print(response2.get("result"))


HYBRID RAG (dense + BM25, weights=[0.3, 0.7]):
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Sun Also Rises”. Similarly, document 1 dominates the posterior when “A Farewell to Arms” is
generated. Intriguingly, after the ﬁrst token of each book is generated, the document posterior ﬂattens.

the parameters of a language model? arXiv e-prints, 2020. URL https://arxiv.org/abs/
2002.08910.
[53] Stephen Robertson and Hugo Zaragoza. The probabilistic relevance framework: Bm25 and

retriever, and then the generator produces a distribution for the next output token for each document,

RAG-Token The RAG-Token model can be seen as a standard, autoregressive seq2seq genera-
tor with transition probability: p0
✓(yi|x, y1:i  1)= P
z2top-k(p(·|x)) p⌘ (zi|x)p✓(yi|x, zi,y 1:i  1) To

distribution over generated text. In one approach, RAG-Sequence, the model uses the same document
to pre

#### Test Query 2: Abstractive Question Answering
> This query contains the specific technical term `abstractive` — BM25 will find it precisely.


In [23]:
query2 = "What is Abstractive Question Answering?"
print(f"Query: '{query2}'\n")
print("=" * 60)
print("NAIVE RAG:")
print("=" * 60)
response3 = normal_chain.invoke(query2)
print(response3.get("result"))


Query: 'What is Abstractive Question Answering?'

NAIVE RAG:
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

3.2 Abstractive Question Answering
RAG models can go beyond simple extractive QA and answer questions with free-form, abstractive

even when the correct answer is not in any retrieved document, achieving 11.8% accuracy in such
cases for NQ, where an extractive model would score 0%.
4.2 Abstractive Question Answering

the popular extractive QA paradigm [ 5, 7, 31, 26], where answers are extracted spans from retrieved
documents, relying primarily on non-parametric knowledge. We also compare to “Closed-Book

Question: What is Abstractive Question Answering?
Helpful Answer: Abstractive Question Answering is a type of question answering that goes beyond simple extractive QA and provides free-form, abstractive answers, even when the correct answer is not in any retrieved

In [24]:
print("=" * 60)
print("HYBRID RAG:")
print("=" * 60)
response4 = hybrid_chain.invoke(query2)
print(response4.get("result"))


HYBRID RAG:
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

3.2 Abstractive Question Answering
RAG models can go beyond simple extractive QA and answer questions with free-form, abstractive

even when the correct answer is not in any retrieved document, achieving 11.8% accuracy in such
cases for NQ, where an extractive model would score 0%.
4.2 Abstractive Question Answering

data. We now discuss experimental details for each task.
3.1 Open-domain Question Answering
Open-domain question answering (QA) is an important real-world application and common testbed

the popular extractive QA paradigm [ 5, 7, 31, 26], where answers are extracted spans from retrieved
documents, relying primarily on non-parametric knowledge. We also compare to “Closed-Book

Question: What is Abstractive Question Answering?
Helpful Answer: Abstractive Question Answering is a type of question answeri

#### 🔍 Inspect Retrieved Documents (What each retriever actually fetched)


In [25]:
# See exactly what context each retriever passes to the LLM
test_query = "What is Abstractive Question Answering?"

dense_docs = dense_retriever.invoke(test_query)
hybrid_docs = ensemble_retriever.invoke(test_query)

print(f"Dense retriever returned {len(dense_docs)} docs:")
for i, doc in enumerate(dense_docs):
    print(f"  [{i}] {doc.page_content[:120]}...")

print(f"\nEnsemble retriever returned {len(hybrid_docs)} docs:")
for i, doc in enumerate(hybrid_docs):
    print(f"  [{i}] {doc.page_content[:120]}...")

print(f"\n💡 Hybrid retriever provides {len(hybrid_docs) - len(dense_docs)} additional"
      f" docs from keyword matching — richer context for the LLM.")

Dense retriever returned 3 docs:
  [0] 3.2 Abstractive Question Answering
RAG models can go beyond simple extractive QA and answer questions with free-form, ab...
  [1] even when the correct answer is not in any retrieved document, achieving 11.8% accuracy in such
cases for NQ, where an e...
  [2] the popular extractive QA paradigm [ 5, 7, 31, 26], where answers are extracted spans from retrieved
documents, relying ...

Ensemble retriever returned 4 docs:
  [0] 3.2 Abstractive Question Answering
RAG models can go beyond simple extractive QA and answer questions with free-form, ab...
  [1] even when the correct answer is not in any retrieved document, achieving 11.8% accuracy in such
cases for NQ, where an e...
  [2] data. We now discuss experimental details for each task.
3.1 Open-domain Question Answering
Open-domain question answeri...
  [3] the popular extractive QA paradigm [ 5, 7, 31, 26], where answers are extracted spans from retrieved
documents, relying ...

💡 Hybrid retriever 

---
# 🎯 — `Key Takeaways`
---


| # | Takeaway |
|---|----------|
| 01 | Naive retrieval (dense-only) misses exact keyword matches — hybrid closes that gap |
| 02 | TF-IDF and BM25 create sparse vectors — old but powerful for precise technical terms |
| 03 | Ensemble Retriever blends scores via alpha weighting — tune `weights` per your domain |
| 04 | Hybrid RAG gives the LLM richer context → more accurate and grounded answers |
| 05 | LangChain: `BM25Retriever` + `Chroma.as_retriever()` → `EnsembleRetriever` — 4 lines |

### 🔜 Next Episode
**Hybrid Search with Weaviate** + **Cross-Encoder Reranking** — adds a second-stage ranker for even higher precision.

---
**👍 If this helped, like the video, subscribe to the channel, and star the GitHub repo!**  
🔗 GitHub: [mayankchugh-learning](https://github.com/mayankchugh-learning)  
📺 YouTube: [@itaienthusiast](https://youtube.com/@itaienthusiast)  
✍️ Medium: [@mayankchugh.jobathk](https://medium.com/@mayankchugh.jobathk)


In [26]:
pip show langchain

Name: langchain
Version: 1.3.4
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
